# Colab C++ vs CUDA Diffraction Benchmark

Этот блокнот предназначен для **Google Colab** и сравнивает две отдельные native-реализации из GitHub:
- `CPU C++`
- `CUDA C++`

Цель блока: получить воспроизводимые замеры времени, форматный вывод решения, таблицы, графики и интерактивный анализ ускорения по требованиям из `Grafiki.docx`.

## Что покрывает блокнот

1. Явные параметры по умолчанию для решения: `M`, `N`, `λ`, `δ`, границы пластин, угол падения.
2. Форматный вывод решения для `CPU` и `GPU` в случаях без скин-эффекта и со скин-эффектом.
3. Вставки ключевых участков исходников `CPU C++` и `CUDA C++`.
4. Sweep по `N` при фиксированном `M=40` для диапазонов `[5, 100]` и `[50, 500]`.
5. Sweep по `M` при фиксированном `N=10` для диапазона `[10, 100]`.
6. Интерактивные 3D графики сложности по `N` и `M`.
7. Анализ, при каких `N` и `M` `CUDA` становится выгоднее `CPU`.

In [ ]:
from __future__ import annotations

import shutil
import statistics
import subprocess
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    import plotly.graph_objects as go
except Exception:
    go = None

In [ ]:
REPO_URL = 'https://github.com/MaximillianVoss/plate-diffraction-solver.git'
REPO_BRANCH = 'feature/two-plates'
WORKDIR = Path('/content/plate-diffraction-solver')
RESULTS_DIR = WORKDIR / 'output' / 'benchmark-notebook'

DEFAULT_SOLVER_CASE = {
    'alpha1': -1.5,
    'beta1': -0.5,
    'alpha2': 0.5,
    'beta2': 1.5,
    'lambda': 10.0,
    'theta_deg': 10.0,
    'n': 10,
    'm_quad': 40,
    'skin_depth': 0.1,
}

SMALL_N_SWEEP = list(range(5, 101, 5))
LARGE_N_SWEEP = list(range(50, 501, 50))
M_SWEEP = list(range(10, 101, 10))
SURFACE_N_VALUES = list(range(10, 101, 10))
SURFACE_M_VALUES = list(range(10, 101, 10))

REPEATS_SMALL = 2
REPEATS_LARGE = 1
REPEATS_M_SWEEP = 2
REPEATS_SURFACE = 1

print('DEFAULT_SOLVER_CASE =', DEFAULT_SOLVER_CASE)

In [ ]:
def run_command(command, cwd: Path | None = None, check: bool = True):
    completed = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        capture_output=True,
        text=True,
        shell=False,
    )
    if check and completed.returncode != 0:
        raise RuntimeError(
            f'Command failed: {command}\nSTDOUT:\n{completed.stdout}\nSTDERR:\n{completed.stderr}'
        )
    return completed


def ensure_repo():
    if WORKDIR.exists():
        run_command(['git', '-C', str(WORKDIR), 'fetch', 'origin'])
        run_command(['git', '-C', str(WORKDIR), 'checkout', REPO_BRANCH])
        run_command(['git', '-C', str(WORKDIR), 'pull', '--ff-only', 'origin', REPO_BRANCH])
    else:
        run_command(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(WORKDIR)])
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    return WORKDIR


def detect_cuda_arch(default='75'):
    if shutil.which('nvidia-smi') is None:
        return None

    result = run_command(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        check=False,
    )
    if result.returncode != 0:
        return None

    value = result.stdout.strip().splitlines()[0].strip()
    if not value:
        return default
    return value.replace('.', '')

In [ ]:
repo_root = ensure_repo()
print('repo_root =', repo_root)
print('git commit =', run_command(['git', '-C', str(repo_root), 'rev-parse', '--short', 'HEAD']).stdout.strip())

CPU_BUILD_SCRIPT = repo_root / 'Diffraction.Cpp' / 'build_cpu.sh'
CPU_EXE = repo_root / 'Diffraction.Cpp' / 'build' / 'DiffractionCpu'
CUDA_BUILD_SCRIPT = repo_root / 'Diffraction.Cuda' / 'build_cuda.sh'
CUDA_EXE = repo_root / 'Diffraction.Cuda' / 'build' / 'DiffractionCuda'

cpu_build = run_command(['bash', str(CPU_BUILD_SCRIPT)], cwd=repo_root)
print(cpu_build.stdout or cpu_build.stderr)

cuda_arch = detect_cuda_arch()
cuda_available = cuda_arch is not None
print('cuda_available =', cuda_available, 'cuda_arch =', cuda_arch)

if cuda_available:
    cuda_build = run_command(['bash', str(CUDA_BUILD_SCRIPT), cuda_arch], cwd=repo_root, check=False)
    print(cuda_build.stdout or cuda_build.stderr)
    cuda_available = cuda_build.returncode == 0 and CUDA_EXE.exists()
    if not cuda_available:
        print('CUDA build failed, notebook will continue with CPU only.')
else:
    print('GPU runtime not detected, notebook will continue with CPU only.')

## Ключевые участки исходников

Ниже блокнот автоматически подставляет характерные фрагменты `CPU C++` и `CUDA C++`, чтобы можно было быстро сравнить постановку задачи, ввод параметров и вычислительное ядро.

In [ ]:
def extract_snippet(path: Path, start_marker: str, end_marker: str | None = None, max_lines: int = 80):
    lines = path.read_text(encoding='utf-8').splitlines()
    start_index = next(i for i, line in enumerate(lines) if start_marker in line)
    if end_marker is None:
        end_index = min(len(lines), start_index + max_lines)
    else:
        end_index = next(i for i, line in enumerate(lines[start_index + 1:], start_index + 1) if end_marker in line)
    return '\n'.join(lines[start_index:end_index])


def show_source_snippets(root: Path):
    snippets = [
        ('CPU C++: разбор параметров CLI', root / 'Diffraction.Cpp' / 'src' / 'DiffractionCpu.cpp', 'SolverParameters parse_arguments', 'void validate_parameters'),
        ('CPU C++: сборка матрицы', root / 'Diffraction.Cpp' / 'src' / 'DiffractionCpu.cpp', 'void assemble_matrix_cpu(', 'std::vector<ComplexValue> assemble_rhs_cpu'),
        ('CUDA C++: разбор параметров CLI', root / 'Diffraction.Cuda' / 'src' / 'DiffractionCuda.cu', 'SolverParameters parse_arguments', 'void validate_parameters'),
        ('CUDA C++: kernel сборки матрицы', root / 'Diffraction.Cuda' / 'src' / 'DiffractionCuda.cu', '__global__ void assemble_matrix_kernel(', '__global__ void assemble_rhs_kernel'),
    ]

    for title, path, start_marker, end_marker in snippets:
        snippet = extract_snippet(path, start_marker, end_marker)
        display(Markdown(f'### {title}\n```cpp\n{snippet}\n```'))


show_source_snippets(repo_root)

In [ ]:
FLOAT_KEYS = {
    'alpha1', 'beta1', 'alpha2', 'beta2', 'lambda',
    'theta_rad', 'theta_deg', 'skin_depth',
    'chi_re', 'chi_im',
    'assembly_ms', 'solve_ms', 'total_ms',
}
INT_KEYS = {'n', 'm_quad'}


def parse_solver_output(text: str):
    parsed = {
        'status': None,
        'backend': None,
        'message': None,
        'coefficients': [],
    }
    coeff_map = {}

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip()

        if key.startswith('coeff_'):
            idx = int(key.split('_', 1)[1])
            re_text, im_text = value.split(',')
            coeff_map[idx] = complex(float(re_text), float(im_text))
        elif key in FLOAT_KEYS:
            parsed[key] = float(value)
        elif key in INT_KEYS:
            parsed[key] = int(value)
        else:
            parsed[key] = value

    parsed['coefficients'] = [coeff_map[i] for i in sorted(coeff_map)]
    return parsed


def build_solver_args(case: dict):
    return [
        '--alpha1', str(case['alpha1']),
        '--beta1', str(case['beta1']),
        '--alpha2', str(case['alpha2']),
        '--beta2', str(case['beta2']),
        '--lambda', str(case['lambda']),
        '--theta-deg', str(case['theta_deg']),
        '--n', str(case['n']),
        '--m-quad', str(case['m_quad']),
        '--skin-depth', str(case['skin_depth']),
    ]


def run_solver(executable: Path, case: dict):
    result = run_command([str(executable), *build_solver_args(case)], check=False)
    combined_text = result.stdout + ('\n' + result.stderr if result.stderr else '')
    parsed = parse_solver_output(combined_text)
    parsed['returncode'] = result.returncode
    parsed['stdout'] = result.stdout
    parsed['stderr'] = result.stderr
    return parsed


def max_coeff_diff(cpu_coeffs, gpu_coeffs):
    if not cpu_coeffs or not gpu_coeffs or len(cpu_coeffs) != len(gpu_coeffs):
        return None
    return max(abs(a - b) for a, b in zip(cpu_coeffs, gpu_coeffs))


def complex_text(value: complex):
    return f'{value.real:.6f}{value.imag:+.6f}i'


def display_solver_report(result: dict, case: dict, title: str):
    chi_re = result.get('chi_re', 0.0)
    chi_im = result.get('chi_im', 0.0)
    header = (
        f'### {title}\n\n'
        f'**Backend:** `{result.get("backend", "n/a")}`  \n'
        f'**Параметры:** `alpha1={case["alpha1"]}`, `beta1={case["beta1"]}`, `alpha2={case["alpha2"]}`, `beta2={case["beta2"]}`, `λ={case["lambda"]}`, `θ={case["theta_deg"]}°`, `N={case["n"]}`, `M={case["m_quad"]}`, `δ={case["skin_depth"]}`  \n'
        f'**Импедансный коэффициент χ:** `{chi_re:.6f} {chi_im:+.6f}i`  \n'
        f'**Время:** matrix=`{result.get("assembly_ms", float("nan")):.3f} мс`, solve=`{result.get("solve_ms", float("nan")):.3f} мс`, total=`{result.get("total_ms", float("nan")):.3f} мс`'
    )
    display(Markdown(header))

    coeffs = result.get('coefficients', [])
    if coeffs:
        coeff_df = pd.DataFrame({
            'index': list(range(len(coeffs))),
            'value': [complex_text(value) for value in coeffs],
        })
        display(coeff_df)
    else:
        print(result.get('stdout') or result.get('stderr') or 'No coefficients returned.')

## Форматный вывод решения по умолчанию

Этот раздел показывает результат для двух случаев:
- идеальный проводник (`δ = 0`)
- случай со скин-эффектом (`δ = δ_default`)

Ниже один и тот же default-case прогоняется на `CPU`, а при наличии GPU ещё и на `CUDA`.

In [ ]:
DEFAULT_IDEAL_CASE = dict(DEFAULT_SOLVER_CASE)
DEFAULT_IDEAL_CASE['skin_depth'] = 0.0
DEFAULT_SKIN_CASE = dict(DEFAULT_SOLVER_CASE)

cpu_ideal = run_solver(CPU_EXE, DEFAULT_IDEAL_CASE)
cpu_skin = run_solver(CPU_EXE, DEFAULT_SKIN_CASE)

display_solver_report(cpu_ideal, DEFAULT_IDEAL_CASE, 'CPU: идеальный проводник')
display_solver_report(cpu_skin, DEFAULT_SKIN_CASE, 'CPU: случай со скин-эффектом')

if cuda_available:
    cuda_ideal = run_solver(CUDA_EXE, DEFAULT_IDEAL_CASE)
    cuda_skin = run_solver(CUDA_EXE, DEFAULT_SKIN_CASE)
    display_solver_report(cuda_ideal, DEFAULT_IDEAL_CASE, 'CUDA: идеальный проводник')
    display_solver_report(cuda_skin, DEFAULT_SKIN_CASE, 'CUDA: случай со скин-эффектом')
    print('max_coeff_abs_diff, ideal =', max_coeff_diff(cpu_ideal['coefficients'], cuda_ideal['coefficients']))
    print('max_coeff_abs_diff, skin  =', max_coeff_diff(cpu_skin['coefficients'], cuda_skin['coefficients']))
else:
    print('CUDA runtime not available, GPU report skipped.')

In [ ]:
def benchmark_backend(executable: Path, case: dict, repeats: int):
    runs = [run_solver(executable, case) for _ in range(repeats)]
    last = runs[-1]
    return {
        'status': last.get('status'),
        'backend': last.get('backend'),
        'assembly_ms_mean': statistics.mean(run['assembly_ms'] for run in runs),
        'solve_ms_mean': statistics.mean(run['solve_ms'] for run in runs),
        'total_ms_mean': statistics.mean(run['total_ms'] for run in runs),
        'coefficients': last.get('coefficients', []),
        'chi_re': last.get('chi_re'),
        'chi_im': last.get('chi_im'),
    }


def benchmark_cases(cases: list[dict], study_name: str, repeats: int):
    rows = []
    for index, case in enumerate(cases, start=1):
        print(f'[{study_name}] {index}/{len(cases)} -> N={case["n"]}, M={case["m_quad"]}, skin={case["skin_depth"]}, theta={case["theta_deg"]}')
        cpu = benchmark_backend(CPU_EXE, case, repeats)
        rows.append({
            **case,
            'study': study_name,
            'backend': 'CPU C++',
            'assembly_ms_mean': cpu['assembly_ms_mean'],
            'solve_ms_mean': cpu['solve_ms_mean'],
            'total_ms_mean': cpu['total_ms_mean'],
            'status': cpu['status'],
            'chi_re': cpu['chi_re'],
            'chi_im': cpu['chi_im'],
            'max_coeff_abs_diff_vs_cpu': 0.0,
        })

        if cuda_available:
            cuda = benchmark_backend(CUDA_EXE, case, repeats)
            rows.append({
                **case,
                'study': study_name,
                'backend': 'CUDA',
                'assembly_ms_mean': cuda['assembly_ms_mean'],
                'solve_ms_mean': cuda['solve_ms_mean'],
                'total_ms_mean': cuda['total_ms_mean'],
                'status': cuda['status'],
                'chi_re': cuda['chi_re'],
                'chi_im': cuda['chi_im'],
                'max_coeff_abs_diff_vs_cpu': max_coeff_diff(cpu['coefficients'], cuda['coefficients']),
            })
    return pd.DataFrame(rows)


def merge_cpu_cuda(df: pd.DataFrame):
    if df.empty or 'CUDA' not in set(df['backend']):
        return None

    keys = ['study', 'alpha1', 'beta1', 'alpha2', 'beta2', 'lambda', 'theta_deg', 'n', 'm_quad', 'skin_depth']
    cpu_df = df[df['backend'] == 'CPU C++'].copy()
    cuda_df = df[df['backend'] == 'CUDA'].copy()
    merged = cpu_df.merge(cuda_df, on=keys, suffixes=('_cpu', '_cuda'))
    merged['speedup_assembly_cpu_over_cuda'] = merged['assembly_ms_mean_cpu'] / merged['assembly_ms_mean_cuda']
    merged['speedup_solve_cpu_over_cuda'] = merged['solve_ms_mean_cpu'] / merged['solve_ms_mean_cuda']
    merged['speedup_total_cpu_over_cuda'] = merged['total_ms_mean_cpu'] / merged['total_ms_mean_cuda']
    return merged


def save_dataframe(df: pd.DataFrame, name: str):
    path = RESULTS_DIR / name
    df.to_csv(path, index=False)
    print('Saved:', path)
    return path

## Обозначения столбцов в таблицах

- `assembly_ms_mean`: среднее время сборки матрицы.
- `solve_ms_mean`: среднее время решения СЛАУ.
- `total_ms_mean`: среднее полное время backend-а.
- `max_coeff_abs_diff_vs_cpu`: максимальное отклонение коэффициентов относительно `CPU C++`.
- `speedup_*`: ускорение по формуле `CPU_time / CUDA_time`; значение больше `1` означает выигрыш `CUDA`.

In [ ]:
def display_study_table(df: pd.DataFrame, title: str, sort_cols: list[str], columns: list[str]):
    display(Markdown(f'### {title}'))
    display(df.sort_values(sort_cols)[columns].reset_index(drop=True))


def plot_operation_breakdown(df: pd.DataFrame, x_col: str, title: str):
    backends = list(df['backend'].unique())
    fig, axes = plt.subplots(1, len(backends), figsize=(6 * len(backends), 4), sharey=True)
    if len(backends) == 1:
        axes = [axes]

    for ax, backend in zip(axes, backends):
        subset = df[df['backend'] == backend].sort_values(x_col)
        ax.bar(subset[x_col], subset['assembly_ms_mean'], label='matrix')
        ax.bar(subset[x_col], subset['solve_ms_mean'], bottom=subset['assembly_ms_mean'], label='solve')
        ax.plot(subset[x_col], subset['total_ms_mean'], color='black', marker='o', label='total')
        ax.set_title(backend)
        ax.set_xlabel(x_col)
        ax.set_ylabel('ms')
        ax.grid(True, alpha=0.3)
        ax.legend()
    fig.suptitle(title, y=1.04)
    plt.tight_layout()
    plt.show()


def plot_metric_compare(df: pd.DataFrame, x_col: str, title: str):
    metrics = [
        ('assembly_ms_mean', 'Matrix build time'),
        ('solve_ms_mean', 'Linear solve time'),
        ('total_ms_mean', 'Total backend time'),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, (metric, metric_title) in zip(axes, metrics):
        for backend in df['backend'].unique():
            subset = df[df['backend'] == backend].sort_values(x_col)
            ax.plot(subset[x_col], subset[metric], marker='o', label=backend)
        ax.set_title(metric_title)
        ax.set_xlabel(x_col)
        ax.set_ylabel('ms')
        ax.grid(True, alpha=0.3)
        ax.legend()
    fig.suptitle(title, y=1.04)
    plt.tight_layout()
    plt.show()


def plot_speedup(merged: pd.DataFrame, x_col: str, title: str):
    if merged is None or merged.empty:
        print('CUDA data not available, speedup plot skipped.')
        return
    metrics = [
        ('speedup_assembly_cpu_over_cuda', 'Matrix build speedup'),
        ('speedup_solve_cpu_over_cuda', 'Solve speedup'),
        ('speedup_total_cpu_over_cuda', 'Total speedup'),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, (metric, metric_title) in zip(axes, metrics):
        subset = merged.sort_values(x_col)
        ax.plot(subset[x_col], subset[metric], marker='o')
        ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
        ax.set_title(metric_title)
        ax.set_xlabel(x_col)
        ax.set_ylabel('CPU / CUDA')
        ax.grid(True, alpha=0.3)
    fig.suptitle(title, y=1.04)
    plt.tight_layout()
    plt.show()

## Исследование 1. Sweep по `N` при фиксированном `M = 40`, диапазон `[5, 100]`

Ниже строятся:
- таблица со значениями времени по `CPU` и `CUDA`
- график распределения времени по операциям
- сравнение `matrix / solve / total`

In [ ]:
small_n_cases = []
for n_value in SMALL_N_SWEEP:
    case = dict(DEFAULT_SOLVER_CASE)
    case['n'] = n_value
    case['m_quad'] = 40
    small_n_cases.append(case)

small_n_df = benchmark_cases(small_n_cases, 'small_n', REPEATS_SMALL)
save_dataframe(small_n_df, 'study_small_n.csv')
small_n_merged = merge_cpu_cuda(small_n_df)

display_study_table(
    small_n_df,
    'Таблица 1. Времена для sweep по N, диапазон [5, 100], M=40',
    ['backend', 'n'],
    ['backend', 'n', 'm_quad', 'assembly_ms_mean', 'solve_ms_mean', 'total_ms_mean', 'max_coeff_abs_diff_vs_cpu']
)

plot_operation_breakdown(small_n_df, 'n', 'Распределение времени по операциям vs N, M=40')
plot_metric_compare(small_n_df, 'n', 'Сравнение времен CPU и CUDA vs N, M=40')

## Исследование 2. Sweep по `N` при фиксированном `M = 40`, диапазон `[50, 500]`

Это тот же анализ, но уже для больших размерностей. По нему дальше строится график эффективности ускорения `GPU`.

In [ ]:
large_n_cases = []
for n_value in LARGE_N_SWEEP:
    case = dict(DEFAULT_SOLVER_CASE)
    case['n'] = n_value
    case['m_quad'] = 40
    large_n_cases.append(case)

large_n_df = benchmark_cases(large_n_cases, 'large_n', REPEATS_LARGE)
save_dataframe(large_n_df, 'study_large_n.csv')
large_n_merged = merge_cpu_cuda(large_n_df)

display_study_table(
    large_n_df,
    'Таблица 2. Времена для sweep по N, диапазон [50, 500], M=40',
    ['backend', 'n'],
    ['backend', 'n', 'm_quad', 'assembly_ms_mean', 'solve_ms_mean', 'total_ms_mean', 'max_coeff_abs_diff_vs_cpu']
)

plot_operation_breakdown(large_n_df, 'n', 'Распределение времени по операциям vs N, диапазон [50, 500], M=40')
plot_metric_compare(large_n_df, 'n', 'Сравнение времен CPU и CUDA vs N, диапазон [50, 500], M=40')
plot_speedup(large_n_merged, 'n', 'Эффективность ускорения CUDA vs N, диапазон [50, 500], M=40')

## Исследование 3. Sweep по `M` при фиксированном `N = 10`, диапазон `[10, 100]`

Ниже выводятся таблица, график сравнения времени построения матрицы и график ускорения.

In [ ]:
m_sweep_cases = []
for m_value in M_SWEEP:
    case = dict(DEFAULT_SOLVER_CASE)
    case['n'] = 10
    case['m_quad'] = m_value
    m_sweep_cases.append(case)

m_sweep_df = benchmark_cases(m_sweep_cases, 'm_sweep', REPEATS_M_SWEEP)
save_dataframe(m_sweep_df, 'study_m_sweep.csv')
m_sweep_merged = merge_cpu_cuda(m_sweep_df)

display_study_table(
    m_sweep_df,
    'Таблица 3. Времена для sweep по M, диапазон [10, 100], N=10',
    ['backend', 'm_quad'],
    ['backend', 'n', 'm_quad', 'assembly_ms_mean', 'solve_ms_mean', 'total_ms_mean', 'max_coeff_abs_diff_vs_cpu']
)

plot_metric_compare(m_sweep_df, 'm_quad', 'Сравнение времен CPU и CUDA vs M, N=10')
plot_speedup(m_sweep_merged, 'm_quad', 'Эффективность ускорения CUDA vs M, N=10')

## Исследование 4. 3D сложность по `N` и `M`

В этом блоке строятся поверхности для `CPU`, `CUDA` и `speedup`.
По требованию документa графики делаются интерактивными: можно вращать, приближать и наводить курсор.

In [ ]:
surface_cases = []
for n_value in SURFACE_N_VALUES:
    for m_value in SURFACE_M_VALUES:
        case = dict(DEFAULT_SOLVER_CASE)
        case['n'] = n_value
        case['m_quad'] = m_value
        surface_cases.append(case)

surface_df = benchmark_cases(surface_cases, 'surface_nm', REPEATS_SURFACE)
save_dataframe(surface_df, 'study_surface_nm.csv')
surface_merged = merge_cpu_cuda(surface_df)

display_study_table(
    surface_df,
    'Таблица 4. Поверхность сложности по N и M',
    ['backend', 'n', 'm_quad'],
    ['backend', 'n', 'm_quad', 'assembly_ms_mean', 'solve_ms_mean', 'total_ms_mean', 'max_coeff_abs_diff_vs_cpu']
)

In [ ]:
def build_surface(dataframe: pd.DataFrame, value_col: str):
    pivot = dataframe.pivot(index='m_quad', columns='n', values=value_col).sort_index().sort_index(axis=1)
    x = pivot.columns.to_numpy(dtype=float)
    y = pivot.index.to_numpy(dtype=float)
    z = pivot.to_numpy(dtype=float)
    return x, y, z


def show_surface(dataframe: pd.DataFrame, value_col: str, title: str):
    if dataframe is None or dataframe.empty:
        print(f'No data for {title}')
        return

    x, y, z = build_surface(dataframe, value_col)
    if go is None:
        fig = plt.figure(figsize=(7, 5))
        ax = fig.add_subplot(111, projection='3d')
        xx, yy = np.meshgrid(x, y)
        ax.plot_surface(xx, yy, z, cmap='viridis')
        ax.set_title(title)
        ax.set_xlabel('N')
        ax.set_ylabel('M')
        ax.set_zlabel(value_col)
        plt.show()
        return

    fig = go.Figure(data=[go.Surface(x=x, y=y, z=z, colorscale='Viridis')])
    fig.update_layout(
        title=title,
        scene=dict(xaxis_title='N', yaxis_title='M', zaxis_title=value_col),
        width=900,
        height=650,
    )
    fig.show()


cpu_surface = surface_df[surface_df['backend'] == 'CPU C++'].copy()
show_surface(cpu_surface, 'total_ms_mean', 'CPU C++: total time surface over N and M')
show_surface(cpu_surface, 'solve_ms_mean', 'CPU C++: solve time surface over N and M')

if cuda_available:
    cuda_surface = surface_df[surface_df['backend'] == 'CUDA'].copy()
    show_surface(cuda_surface, 'total_ms_mean', 'CUDA: total time surface over N and M')
    show_surface(cuda_surface, 'solve_ms_mean', 'CUDA: solve time surface over N and M')

if surface_merged is not None:
    show_surface(surface_merged, 'speedup_total_cpu_over_cuda', 'CUDA speedup by total time over N and M')
    show_surface(surface_merged, 'speedup_solve_cpu_over_cuda', 'CUDA speedup by solve time over N and M')

## Анализ выгодности `CUDA`

Ниже строится аналитическая таблица: для каждого `M` ищется минимальный `N`, начиная с которого `CUDA` даёт ускорение больше `1`.

In [ ]:
def first_profitable_n(merged: pd.DataFrame, speedup_column: str):
    records = []
    if merged is None or merged.empty:
        return pd.DataFrame(records)

    for m_value, part in merged.groupby('m_quad'):
        profitable = part[part[speedup_column] > 1.0].sort_values('n')
        records.append({
            'm_quad': m_value,
            speedup_column: profitable.iloc[0]['n'] if not profitable.empty else np.nan,
        })
    return pd.DataFrame(records)


if surface_merged is not None:
    profit_total = first_profitable_n(surface_merged, 'speedup_total_cpu_over_cuda')
    profit_solve = first_profitable_n(surface_merged, 'speedup_solve_cpu_over_cuda')
    profit_assembly = first_profitable_n(surface_merged, 'speedup_assembly_cpu_over_cuda')

    profitability = profit_total.merge(profit_solve, on='m_quad').merge(profit_assembly, on='m_quad')
    profitability.columns = [
        'm_quad',
        'first_n_total_speedup_gt_1',
        'first_n_solve_speedup_gt_1',
        'first_n_assembly_speedup_gt_1',
    ]

    display(Markdown('### Таблица 5. Минимальный N, начиная с которого CUDA становится выгодной'))
    display(profitability)
else:
    print('Profitability analysis skipped: CUDA data not available.')

## Что именно показывают таблицы и графики

- **Таблица 1**: времена `CPU/CUDA` для `N in [5, 100]`, фиксированный `M=40`.
- **Таблица 2**: те же метрики, но для `N in [50, 500]`.
- **Таблица 3**: времена для sweep по `M` при `N=10`.
- **Таблица 4**: полная поверхность по `N` и `M`.
- **Таблица 5**: порог выгодности `CUDA` по `N` для каждого `M`.
- **Графики распределения по операциям**: как полное время раскладывается на `matrix build` и `solve`.
- **Графики сравнения времени**: прямое сравнение `assembly / solve / total` между `CPU` и `CUDA`.
- **Графики speedup**: отношение `CPU_time / CUDA_time`.
- **3D поверхности**: интерактивная карта сложности и ускорения в пространстве `N × M`.

## Дальше

Если потребуется расширить исследование, логично следующим шагом добавить в native backend'ы ещё и метрики точности: ошибку ГУ, невязку Гельмгольца и энергетику. Тогда тот же блокнот можно будет использовать не только для сравнения производительности, но и для контроля физической достоверности решения.